# Six Nations 2026 — Final Week (Week 5)

**Date:** 2026-03-11

This notebook covers:
1. **Week 4 retrospective** — how did the model's predictions fare? (spoiler: all three correct!)
2. **Current standings** — where do we stand after four rounds
3. **Week 5 predictions** — the final weekend of matches on 14 March
4. **Paths to victory** — the various scenarios for each team's final position
5. **Predicted final standings** — probabilistic projection of the title race

In [1]:
import sys
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

sys.path.insert(0, '/home/daniel/repositories/personal/rugby-ranking')

from rugby_ranking.notebook_utils import setup_notebook_environment, load_model_and_trace
from rugby_ranking.model.predictions import MatchPredictor
from rugby_ranking.model.data_utils import prepare_season_data, quick_standings
from rugby_ranking.model.season_predictor import SeasonPredictor, BonusPointRules
from rugby_ranking.model.paths_to_victory import PathsAnalyzer

# --------------------------------------------------------------------------- #
# Data loading
# --------------------------------------------------------------------------- #
dataset, df, model_dir = setup_notebook_environment()

Loading from: /home/daniel/repositories/personal/Rugby-Data
Loaded 14632 matches from /home/daniel/repositories/personal/Rugby-Data/json
Found 12465 unique players
Found 228 unique teams
✓ Loaded 435,022 player-match observations
  Players: 12,458
  Teams: 141
  Matches: 19,184
  Date range: 2006-09-02 to 2026-03-11

Data Quality Checks:
✓ Removed 1381 conversions and 451 penalties
  from 1038 player-match records in non-kicking positions
  Affected players: 446


In [2]:
model, trace = load_model_and_trace('international-mini5')

ℹ  Loading checkpoint: international-mini5
Loaded checkpoint from /home/daniel/.cache/rugby_ranking/international-mini5
✓ Loaded successfully
ℹ    Players: 7,127
ℹ    Team-seasons: 428


In [3]:
# Colour palette and style
COLORS = {
    'England':  '#C8102E',
    'France':   '#002395',
    'Ireland':  '#009A44',
    'Italy':    '#0066CC',
    'Scotland': '#003F8F',
    'Wales':    '#D62B2F',
}
TEAMS = list(COLORS.keys())

plt.style.use([
    '~/.config/matplotlib/stylelib/kentigern-light.mplstyle',
    '~/.config/matplotlib/stylelib/kentigern-map.mplstyle',
])

# --------------------------------------------------------------------------- #
# Output directories
# --------------------------------------------------------------------------- #
BLOG_DATA_DIR   = '/home/daniel/repositories/websites/blog/data/2026/03/11'
BLOG_IMAGES_DIR = '/home/daniel/repositories/websites/blog/images/2026/03/11'
os.makedirs(BLOG_DATA_DIR,   exist_ok=True)
os.makedirs(BLOG_IMAGES_DIR, exist_ok=True)

predictor = MatchPredictor(model, trace)
print(f'Data dir  : {BLOG_DATA_DIR}')
print(f'Images dir: {BLOG_IMAGES_DIR}')

Data dir  : /home/daniel/repositories/websites/blog/data/2026/03/11
Images dir: /home/daniel/repositories/websites/blog/images/2026/03/11


---
## 1. Week 4 Retrospective

Three games, three correct calls — the model's best week yet.

| Prediction | Actual |
|:---|:---|
| **Ireland** 32-28 Wales | **Ireland** 27-17 Wales ✓ |
| **Scotland** 29-25 France | **Scotland** 50-40 France ✓ |
| **Italy** 28-23 England | **Italy** 23-18 England ✓ |

Scotland's win over France was the standout result — a 10-point victory that reshuffled the title race dramatically.
Below we look at how likely each result was according to the model.

In [4]:
def plot_2d_predictions(predictions, truth=None, median=True):
    """2-D KDE heatmap of match score predictions."""
    f, ax = plt.subplots(1, 1, figsize=(6, 6))

    bins = np.arange(0, 100, 1)
    X, Y = np.meshgrid(bins, bins)

    estimator = stats.gaussian_kde(
        np.vstack([predictions.home.samples, predictions.away.samples])
    )
    Z = estimator(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

    ax.set_xlim(0, 50)
    ax.set_ylim(0, 50)

    probs = [0.393, 0.865, 0.989]
    z_sorted = np.sort(Z.ravel())[::-1]
    cdf = np.cumsum(z_sorted)
    cdf /= cdf[-1]
    levels = sorted(float(z_sorted[np.searchsorted(cdf, p)]) for p in probs)

    ax.matshow(Z, extent=(0, 100, 0, 100), origin='lower', cmap='inferno', alpha=0.7)
    cs = ax.contour(X, Y, Z, levels=levels, colors=['white'])
    ax.clabel(cs, fmt={levels[0]: '3σ', levels[1]: '2σ', levels[2]: '1σ'}, inline=True)

    ax.set_xlabel(f'{predictions.home.team} points')
    ax.set_ylabel(f'{predictions.away.team} points')

    if truth:
        ax.scatter(truth[0], truth[1], color='white', marker='o', s=120,
                   zorder=5, label='Actual')

    if median:
        ax.scatter(
            np.median(predictions.home.samples),
            np.median(predictions.away.samples),
            color='black', marker='X', s=120, zorder=6, label='Prediction'
        )

    ax.legend(loc='upper right', fontsize=9)
    return f


def export_kde_grid(predictions, filename, truth=None, grid_max=50, grid_step=1):
    """Export a KDE grid for D3 rendering."""
    bins = list(range(0, grid_max, grid_step))
    X, Y = np.meshgrid(bins, bins)

    estimator = stats.gaussian_kde(
        np.vstack([predictions.home.samples, predictions.away.samples])
    )
    Z = estimator(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

    z_sorted = np.sort(Z.ravel())[::-1]
    cdf = np.cumsum(z_sorted)
    cdf /= cdf[-1]
    probs_list = [0.393, 0.865, 0.989]
    levels = sorted(float(z_sorted[np.searchsorted(cdf, p)]) for p in probs_list)

    out = {
        'home':     predictions.home.team,
        'away':     predictions.away.team,
        'bins':     bins,
        'grid_max': grid_max,
        'grid_step': grid_step,
        'z':        Z.tolist(),
        'levels':   levels,
        'median': {
            'home': float(np.median(predictions.home.samples)),
            'away': float(np.median(predictions.away.samples)),
        },
    }
    if truth:
        out['truth'] = {'home': truth[0], 'away': truth[1]}

    path = f'{BLOG_DATA_DIR}/{filename}'
    with open(path, 'w') as fh:
        json.dump(out, fh)
    print(f'Exported {path}')
    return out

In [5]:
# ── Week 4 actual results ──────────────────────────────────────────────────────
WEEK4_RESULTS = [
    ('Ireland', 'Wales',    (27, 17)),
    ('Scotland', 'France',  (50, 40)),
    ('Italy',    'England', (23, 18)),
]

for home, away, (hs, as_) in WEEK4_RESULTS:
    preds = predictor.predict_teams_only(home, away, '2026-2027')

    # PNG for blog
    slug = f'{home[:3].lower()}-{away[:3].lower()}'
    for theme in ('light', 'dark'):
        with plt.style.context(f'~/.config/matplotlib/stylelib/kentigern-{theme}.mplstyle'):
            f = plot_2d_predictions(preds, truth=(hs, as_))
            f.savefig(f'{BLOG_IMAGES_DIR}/week4-{slug}-{theme}.png', dpi=300, bbox_inches='tight')
            plt.close(f)

    # JSON for D3
    export_kde_grid(preds, f'week4-{slug}.json', truth=(hs, as_))
    print(f'  {home} {hs}–{as_} {away}  |  predicted median: '
          f'{np.median(preds.home.samples):.0f}–{np.median(preds.away.samples):.0f}')

Exported /home/daniel/repositories/websites/blog/data/2026/03/11/week4-ire-wal.json
  Ireland 27–17 Wales  |  predicted median: 25–19
Exported /home/daniel/repositories/websites/blog/data/2026/03/11/week4-sco-fra.json
  Scotland 50–40 France  |  predicted median: 26–21
Exported /home/daniel/repositories/websites/blog/data/2026/03/11/week4-ita-eng.json
  Italy 23–18 England  |  predicted median: 24–20


---
## 2. Current Standings (after Week 4)

In [6]:
from rugby_ranking.model.data_utils import quick_standings
from rugby_ranking.model.league_table import format_table

standings = quick_standings(
    dataset,
    season='2026-2027',
    competition='six-nations',
    bonus_rules='SIX_NATIONS',
)
print(format_table(standings))

 Pos | Team                 |    P |    W |    D |    L |    F |    A |  +/- |   TF |   BP |  Pts
-------------------------------------------------------------------------------------------------
     1 | France               |    4 |    3 |    0 |    1 |  163 |   84 |   79 |   24 |    4 |   16
     2 | Scotland             |    4 |    3 |    0 |    1 |  122 |  101 |   21 |   17 |    4 |   16
     3 | Ireland              |    4 |    3 |    0 |    1 |  103 |   87 |   16 |   14 |    2 |   14
     4 | Italy                |    4 |    2 |    0 |    2 |   62 |   86 |  -24 |    6 |    1 |    9
     5 | England              |    4 |    1 |    0 |    3 |  107 |  103 |    4 |   14 |    2 |    6
     6 | Wales                |    4 |    0 |    0 |    4 |   59 |  155 |  -96 |    7 |    1 |    1


---
## 3. Recent Form

In [7]:
INTERNATIONAL_COMPS = [
    'six-nations', 'mid-year-internationals',
    'end-of-year-internationals', 'world-cup',
]

match_df = (
    df[df['competition'].isin(INTERNATIONAL_COMPS)]
    .drop_duplicates(subset=['team', 'date', 'opponent'])
    .copy()
)
match_df['date']   = pd.to_datetime(match_df['date'])
match_df['margin'] = match_df['team_score'] - match_df['opponent_score']
print(f'International match records: {len(match_df)}')

International match records: 557


In [8]:
RESULT_COLORS = {'win': '#2ecc71', 'loss': '#e74c3c', 'draw': '#f39c12'}


def plot_recent_form(team, n=15, ax=None):
    """Horizontal strip showing last n test match results."""
    team_matches = match_df[match_df['team'] == team].sort_values('date').tail(n)
    if ax is None:
        f, ax = plt.subplots(1, 1, figsize=(10, 1.5))

    for i, (_, row) in enumerate(team_matches.iterrows()):
        color = RESULT_COLORS.get(row['match_result'], 'gray')
        ax.barh(0, 0.85, left=i + 0.075, height=0.7, color=color, alpha=0.85)
        opp_abbr = row['opponent'][:3].upper()
        ax.text(i + 0.5,  0.18, opp_abbr,
                ha='center', va='center', fontsize=8, fontweight='bold', color='white')
        ax.text(i + 0.5, -0.20,
                f"{int(row['team_score'])}-{int(row['opponent_score'])}",
                ha='center', va='center', fontsize=7, color='white')

    ax.set_xlim(0, n)
    ax.set_ylim(-0.55, 0.55)
    ax.axis('off')
    ax.set_title(f'{team}  (older → newer)', loc='left', fontsize=10)


for theme in ('light', 'dark'):
    with plt.style.context(f'~/.config/matplotlib/stylelib/kentigern-{theme}.mplstyle'):
        f, axes = plt.subplots(6, 1, figsize=(10, 5.5),
                               gridspec_kw={'hspace': 0.8})
        f.suptitle('Recent form going into Week 5 (Final Round)', fontsize=13, y=1.02)
        for ax, team in zip(axes, TEAMS):
            plot_recent_form(team, ax=ax, n=15)
        plt.tight_layout()
        f.savefig(f'{BLOG_IMAGES_DIR}/recent-form-all-{theme}.png',
                  dpi=300, bbox_inches='tight')
        plt.close(f)

/tmp/ipykernel_514/3883011790.py:33: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipykernel_514/3883011790.py:33: UserWarning: The figure layout has changed to tight
  plt.tight_layout()
/tmp/ipykernel_514/3883011790.py:33: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipykernel_514/3883011790.py:33: UserWarning: The figure layout has changed to tight
  plt.tight_layout()


In [9]:
def export_recent_form(teams, n=15, filename='recent-form.json'):
    """Export recent form data to JSON for D3 rendering."""
    out = {}
    for team in teams:
        team_matches = (
            match_df[match_df['team'] == team]
            .sort_values('date')
            .tail(n)
        )
        out[team] = [
            {
                'date':           row['date'].strftime('%Y-%m-%d'),
                'opponent':       row['opponent'],
                'team_score':     int(row['team_score']),
                'opponent_score': int(row['opponent_score']),
                'result':         row['match_result'],
                'margin':         int(row['margin']),
            }
            for _, row in team_matches.iterrows()
        ]

    path = f'{BLOG_DATA_DIR}/{filename}'
    with open(path, 'w') as fh:
        json.dump(out, fh)
    print(f'Exported {path}')


export_recent_form(TEAMS, n=15)

Exported /home/daniel/repositories/websites/blog/data/2026/03/11/recent-form.json


---
## 4. Week 5 (Final Round) Predictions

All three matches on **Saturday 14 March 2026**:

| Home | Away |
|:-----|:-----|
| Ireland  | Scotland |
| Wales    | Italy    |
| France   | England  |

In [10]:
WEEK5_FIXTURES = [
    ('Ireland',  'Scotland'),
    ('Wales',    'Italy'),
    ('France',   'England'),
]

week5_predictions = {}

for home, away in WEEK5_FIXTURES:
    preds = predictor.predict_teams_only(home, away, '2026-2027')
    week5_predictions[(home, away)] = preds

    home_median = int(np.median(preds.home.samples))
    away_median = int(np.median(preds.away.samples))
    home_win_p  = float(np.mean(preds.home.samples > preds.away.samples))

    print(f'{home} vs {away}')
    print(f'  Predicted: {home_median}–{away_median}')
    print(f'  {home} win probability: {home_win_p:.0%}')
    print()

    # PNG for blog
    slug = f'{home[:3].lower()}-{away[:3].lower()}'
    for theme in ('light', 'dark'):
        with plt.style.context(f'~/.config/matplotlib/stylelib/kentigern-{theme}.mplstyle'):
            f = plot_2d_predictions(preds)
            f.savefig(f'{BLOG_IMAGES_DIR}/week5-{slug}-{theme}.png',
                      dpi=300, bbox_inches='tight')
            plt.close(f)

    # JSON for D3
    export_kde_grid(preds, f'week5-{slug}.json')

Ireland vs Scotland
  Predicted: 24–20
  Ireland win probability: 58%

Exported /home/daniel/repositories/websites/blog/data/2026/03/11/week5-ire-sco.json
Wales vs Italy
  Predicted: 23–19
  Wales win probability: 60%

Exported /home/daniel/repositories/websites/blog/data/2026/03/11/week5-wal-ita.json
France vs England
  Predicted: 27–19
  France win probability: 67%

Exported /home/daniel/repositories/websites/blog/data/2026/03/11/week5-fra-eng.json


---
## 5. Season Simulation — Predicted Final Standings

In [11]:
played_matches, remaining_fixtures = prepare_season_data(
    dataset,
    season='2026-2027',
    competition='six-nations',
    include_tries=True,
)

print(f'Played: {len(played_matches) // 2} matches')
print(f'Remaining: {len(remaining_fixtures)} fixtures')
print()
print('Upcoming fixtures:')
print(remaining_fixtures.to_string(index=False))

Played: 12 matches
Remaining: 3 fixtures

Upcoming fixtures:
home_team away_team                      date
  Ireland  Scotland 2026-03-14 14:10:00+00:00
    Wales     Italy 2026-03-14 16:40:00+00:00
   France   England 2026-03-14 21:10:00+00:00


In [12]:
season_predictor = SeasonPredictor(
    match_predictor=predictor,
    competition=BonusPointRules.SIX_NATIONS,
    playoff_spots=0,
)

print('Running Monte Carlo simulation (1000 iterations)...')
season_pred = season_predictor.predict_season(
    played_matches=played_matches,
    remaining_fixtures=remaining_fixtures,
    season='2026-2027',
    n_simulations=1000,
    return_samples=True,
)

print('\nSimulation complete!')
print(season_predictor.format_predictions(season_pred))

Running Monte Carlo simulation (1000 iterations)...

Simulation complete!
SEASON PREDICTION

CURRENT STANDINGS:
----------------------------------------------------------------------
 1. France               P: 4 W: 3 Pts: 16
 2. Scotland             P: 4 W: 3 Pts: 16
 3. Ireland              P: 4 W: 3 Pts: 14
 4. Italy                P: 4 W: 2 Pts:  9
 5. England              P: 4 W: 1 Pts:  6
 6. Wales                P: 4 W: 0 Pts:  1


PREDICTED FINAL STANDINGS:
----------------------------------------------------------------------
 1. France               Pts:19.0 Diff:+86.0
 2. Scotland             Pts:18.0 Diff:+18.0
 3. Ireland              Pts:17.0 Diff:+19.0
 4. Italy                Pts:11.0 Diff:-29.0
 5. England              Pts:8.0 Diff:-3.0
 6. Wales                Pts:4.0 Diff:-91.0


PLAYOFF PROBABILITIES (Top 0):
----------------------------------------------------------------------
France               0.0%
Scotland             0.0%



---
## 6. Position Probability Matrix

In [13]:
import seaborn as sns

pos_probs = season_pred.position_probabilities
# Drop the most_likely_position summary column — only want the P(pos N) columns
prob_cols = [c for c in pos_probs.columns if c.startswith('P(pos')]
pos_probs_plot = pos_probs[prob_cols]

for theme in ('light', 'dark'):
    with plt.style.context(f'~/.config/matplotlib/stylelib/kentigern-{theme}.mplstyle'):
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.heatmap(
            pos_probs_plot,
            annot=True,
            fmt='.0%',
            cmap='RdYlGn',
            vmin=0,
            vmax=1,
            ax=ax,
            cbar_kws={'label': 'Probability'},
        )
        ax.set_title('Six Nations 2026 — Final Position Probabilities', fontsize=13)
        ax.set_xlabel('Final Position')
        ax.set_ylabel('Team')
        fig.savefig(f'{BLOG_IMAGES_DIR}/position-probabilities-{theme}.png',
                    dpi=300, bbox_inches='tight')
        plt.close(fig)

print(pos_probs_plot.to_string())

          P(pos 1)  P(pos 2)  P(pos 3)  P(pos 4)  P(pos 5)  P(pos 6)
France       0.629     0.320     0.051     0.000     0.000       0.0
Scotland     0.173     0.308     0.519     0.000     0.000       0.0
Ireland      0.198     0.372     0.418     0.012     0.000       0.0
Italy        0.000     0.000     0.012     0.780     0.208       0.0
England      0.000     0.000     0.000     0.208     0.792       0.0
Wales        0.000     0.000     0.000     0.000     0.000       1.0


In [14]:
# Export position probabilities to JSON (prob_cols only, no most_likely_position)
pos_data = {
    'teams':     pos_probs_plot.index.tolist(),
    'positions': list(range(1, len(prob_cols) + 1)),
    'probs':     pos_probs_plot.values.tolist(),
}
path = f'{BLOG_DATA_DIR}/position-probabilities.json'
with open(path, 'w') as fh:
    json.dump(pos_data, fh)
print(f'Exported {path}')

Exported /home/daniel/repositories/websites/blog/data/2026/03/11/position-probabilities.json


---
## 7. Paths to Victory (and other positions)

With only three matches remaining, we can enumerate all 8 possible outcome combinations and
weight each by the model's predicted probabilities.
The `PathsAnalyzer` extracts the key conditions under which each team reaches each position.

In [15]:
analyzer = PathsAnalyzer(
    season_prediction=season_pred,
    match_predictor=predictor,
)

In [16]:
# --- Title race: France, Ireland, Scotland -----------------------------------
for team in ['France', 'Ireland', 'Scotland']:
    paths = analyzer.analyze_paths(team=team, target_position=1)
    print(f"{'='*70}")
    print(f"{team}'s path to 1st place: {paths.probability:.1%}")
    print(f"{'='*70}")
    print(paths.narrative)
    print()

IndexError: list index out of range

In [17]:
# --- Scotland paths to 2nd (given France likely wins) -----------------------
sco_2nd = analyzer.analyze_paths(team='Scotland', target_position=2)
print(f"Scotland's path to 2nd place: {sco_2nd.probability:.1%}")
print(sco_2nd.narrative)

IndexError: list index out of range

In [ ]:
ire_2nd = analyzer.analyze_paths(team='Ireland', target_position=2)
print(f"Ireland's path to 2nd place: {ire_2nd.probability:.1%}")
print(ire_2nd.narrative)

In [ ]:
# --- Italy paths to top 4 ---------------------------------------------------
for pos in [3, 4]:
    paths = analyzer.analyze_paths(team='Italy', target_position=pos)
    print(f"Italy's path to position {pos}: {paths.probability:.1%}")
    print(paths.narrative)
    print()

In [ ]:
# --- England: can they salvage a top-4 finish? ------------------------------
for pos in [3, 4, 5]:
    paths = analyzer.analyze_paths(team='England', target_position=pos)
    print(f"England's path to position {pos}: {paths.probability:.1%}")
    if paths.probability > 0.01:
        print(paths.narrative)
    print()

In [ ]:
# Summary table: most likely position for each team
summary_rows = []
for team in TEAMS:
    row = {'team': team}
    for pos in range(1, 7):
        p = analyzer.analyze_paths(team=team, target_position=pos)
        row[f'P{pos}'] = p.probability
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('team')
summary_df.columns = [f'P({i})' for i in range(1, 7)]
print(summary_df.to_string(float_format='{:.1%}'.format))

In [ ]:
# Export paths summary to JSON for blog
paths_data = {
    'teams':     TEAMS,
    'positions': list(range(1, 7)),
    'probs':     summary_df.values.tolist(),
}
path = f'{BLOG_DATA_DIR}/paths-summary.json'
with open(path, 'w') as fh:
    json.dump(paths_data, fh)
print(f'Exported {path}')

---
## 8. Critical Games — Mutual Information

In [ ]:
critical_games = analyzer.find_critical_games(top_n=3)

print('Most Critical Upcoming Matches (by mutual information with final standings):')
print('=' * 70)
key_games_parts = []
for idx, row in critical_games.iterrows():
    home = row['home_team']
    away = row['away_team']
    mi   = row['total_impact']
    print(f"{idx+1}. {home} vs {away}  —  impact: {mi:.3f}")
    key_games_parts.append(f'{home}|{away}|{mi:.3f}|14 Mar')

print()
print('Jekyll include string:')
print('{%% include rugby-key-games.html games="%s" %%}' % ','.join(key_games_parts))

---
## 9. Predicted Final Standings — Jekyll Output

In [ ]:
# Build Jekyll-friendly standings string from season_pred.predicted_standings
# Format: "Team|pts|wins|points_diff|position"
exp = season_pred.predicted_standings
rows_str = ','.join(
    f"{row['team']}|{row['expected_points']}|{row['expected_wins']}|{row['expected_diff']}|{row['predicted_position']}"
    for _, row in exp.iterrows()
)
print('{%% include rugby-standings.html standings="%s" %%}' % rows_str)

In [ ]:
# Build the Jekyll fixture table string for week 5 predictions
fixture_parts = []
for home, away in WEEK5_FIXTURES:
    preds = week5_predictions[(home, away)]
    home_med = int(np.median(preds.home.samples))
    away_med = int(np.median(preds.away.samples))
    home_win_p = float(np.mean(preds.home.samples > preds.away.samples))
    conf = int(round(home_win_p * 100))
    winner = home if home_win_p > 0.5 else away
    if winner == home:
        part = f'{home}|{home_med}|{away_med}|{away}|{conf}|'
    else:
        part = f'{home}|{home_med}|{away_med}|{away}|{100 - conf}|'
    fixture_parts.append(part)

print('Week 5 fixture table (Jekyll):')
print('{%% include rugby-fixture-table.html matches="%s" %%}' % ','.join(fixture_parts))

In [ ]:
# Title race summary for blog narrative
print('=== TITLE RACE SUMMARY ===')
print()
for team in TEAMS:
    p1 = summary_df.loc[team, 'P(1)'] if team in summary_df.index else 0.0
    p2 = summary_df.loc[team, 'P(2)'] if team in summary_df.index else 0.0
    print(f'{team:10s}  P(1st) = {p1:.1%}   P(2nd) = {p2:.1%}')

---
## 10. Blog Narrative Notes

Key talking points to weave into the blog post:

### Week 4 retrospective
- **All three predictions correct** — the model's joint best performance of the tournament.
- Scotland's **50–40 victory over France** was the headline result; the model predicted a close
  Scotland win (29–25) but the margin was emphatic.  The prediction was directionally right but
  France's collapse in the second half was not anticipated.
- Italy's narrow win over England (23–18 vs predicted 28–23) means England finish week 4 with
  only 1 win from 4 games — a stunning underperformance for a side that was pre-tournament
  favourites.

### Title race
- France still lead but their Grand Slam is gone.  They face England in Paris — a fixture they
  should win but must do so to secure the title.
- Ireland host Scotland in what shapes up as the pivotal match of the weekend: the winner
  likely finishes 2nd; the loser potentially drops to 3rd or lower.
- Scotland's momentum is extraordinary — four straight wins after losing to Italy in week 1.

### Lower table
- Italy in 4th is not just possible, it is the *most probable* outcome — a genuine surprise
  given where they started.
- England and Wales fight it out at the bottom; Wales avoid the Wooden Spoon only if Italy
  lose and Wales win.

### Paths to the title
- France need to beat England and hope Ireland–Scotland stays competitive (or Ireland win).
- Scotland need to beat Ireland and hope France slip up against England.
- Ireland need to beat Scotland and hope France lose.

---
## Files exported

| File | Contents |
|:-----|:---------|
| `week4-ire-wal.json` | KDE grid, Ireland vs Wales retrospective |
| `week4-sco-fra.json` | KDE grid, Scotland vs France retrospective |
| `week4-ita-eng.json` | KDE grid, Italy vs England retrospective |
| `week5-ire-sco.json` | KDE grid, Ireland vs Scotland prediction |
| `week5-wal-ita.json` | KDE grid, Wales vs Italy prediction |
| `week5-fra-eng.json` | KDE grid, France vs England prediction |
| `recent-form.json`   | Recent form data for all six teams |
| `position-probabilities.json` | Position probability matrix |
| `paths-summary.json` | Paths-to-position summary |

All PNG images are in `BLOG_IMAGES_DIR` in both light and dark themes.